# 02_模型架构与实现报告

**项目名称**: 基于PaddlePaddle的SASRec序列推荐系统  
**框架**: PaddlePaddle  
**创建日期**: 2024-01-10  

---

## 目录

1. [SASRec模型架构](#1-SASRec模型架构)  
2. [核心组件实现](#2-核心组件实现)  
3. [模型前向传播流程](#3-模型前向传播流程)  
4. [损失函数设计](#4-损失函数设计)  
5. [数据采样策略](#5-数据采样策略)  
6. [训练配置与优化](#6-训练配置与优化)  

---

## 1. SASRec模型架构

### 1.1 自注意力机制理论基础

SASRec的核心是**自注意力机制（Self-Attention）**，它通过计算序列中每个位置与其他位置之间的关联强度，捕捉序列内的长期依赖关系。其数学表达式如下：

**缩放点积注意力（Scaled Dot-Product Attention）**：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

其中：
- $Q \in \mathbb{R}^{n \times d_k}$：查询矩阵（Query）
- $K \in \mathbb{R}^{n \times d_k}$：键矩阵（Key）
- $V \in \mathbb{R}^{n \times d_v}$：值矩阵（Value）
- $d_k$：注意力头的维度
- $\sqrt{d_k}$：缩放因子，用于稳定梯度

**多头注意力（Multi-Head Attention）**：

$$\text{MultiHead}(Q, K, V) = \text{Concat}\left(\text{head}_1, \ldots, \text{head}_h\right)W^O$$

其中每个注意力头计算为：

$$\text{head}_i = \text{Attention}\left(QW_i^Q, KW_i^K, VW_i^V\right)$$

### 1.2 模型概述

SASRec (Self-Attentive Sequential Recommendation) 是基于Transformer的自注意力序列推荐模型，其核心思想是利用自注意力机制对用户历史行为序列进行建模，从而预测用户下一个可能交互的物品。

**核心特点**：
- 使用可学习的位置编码捕捉序列顺序信息
- 通过多头注意力机制学习物品间的依赖关系
- 采用因果掩码确保预测时只能看到历史信息
- 基于物品嵌入向量的内积进行评分预测

### 1.3 模型类定义

模型定义在 `models/sasrec_model.py` 文件中，继承自 `paddle.nn.Layer` 基类。模型初始化时需要指定物品数量、序列最大长度、隐藏层维度、注意力头数、Transformer块数以及Dropout比例等超参数。

In [ ]:
# 模型类定义  models/sasrec_model.py 第16-29行
class SASRec(paddle.nn.Layer):
    def __init__(
        self,
        item_num,
        max_len=50,
        hidden_units=64,
        num_heads=2,
        num_blocks=2,
        dropout_rate=0.5,
    ):
        super(SASRec, self).__init__()
        self.item_num = item_num
        self.max_len = max_len
        self.hidden_units = hidden_units

**超参数说明**：

| 参数 | 含义 | 默认值 |
|------|------|--------|
| item_num | 物品总数 | - |
| max_len | 序列最大长度 | 50 |
| hidden_units | 嵌入和隐藏层维度 | 64 |
| num_heads | 注意力头数 | 2 |
| num_blocks | Transformer层数 | 2 |
| dropout_rate | Dropout比例 | 0.5 |

---

## 2. 核心组件实现

### 2.1 嵌入层

模型包含两个嵌入层：物品嵌入层和位置嵌入层。物品嵌入层将离散的物品ID映射为稠密向量表示，位置嵌入层为序列中的每个位置提供位置信息编码。两者采用加法融合的方式结合。

In [ ]:
# 嵌入层定义 - models/sasrec_model.py 第31-33行
self.item_emb = nn.Embedding(item_num + 1, hidden_units)  # 物品嵌入，+1用于padding
self.pos_emb = nn.Embedding(max_len, hidden_units)        # 位置嵌入
self.emb_dropout = paddle.nn.Dropout(p=dropout_rate)      # 嵌入层Dropout

**嵌入层设计说明**：

- **物品嵌入层**：维度为 `(item_num+1, hidden_units)`，其中索引0作为padding标识，用于填充不足最大长度的序列
- **位置嵌入层**：维度为 `(max_len, hidden_units)`，采用可学习的位置编码方式，而非Transformer原论文中的固定正弦余弦编码
- **Dropout层**：应用于嵌入融合后的结果，用于防止过拟合

### 2.2 位置编码实现

位置编码方法将物品嵌入与位置嵌入相加，为序列中的每个物品赋予位置信息。

In [ ]:
# 位置编码方法 -  models/sasrec_model.py 第47-51行
def position_encoding(self, seqs):
    seqs_embed = self.item_emb(seqs)  # 获取物品嵌入
    positions = np.tile(np.array(range(seqs.shape[1])), [seqs.shape[0], 1])  # 生成位置索引
    position_embed = self.pos_emb(paddle.to_tensor(positions, dtype="int64"))  # 获取位置嵌入
    return self.emb_dropout(seqs_embed + position_embed)  # 加法融合后应用Dropout

**位置编码流程**：

1. 输入序列 `seqs` 形状为 `(batch_size, seq_len)`
2. 通过物品嵌入层得到 `seqs_embed`，形状为 `(batch_size, seq_len, hidden_units)`
3. 生成位置索引数组 `[0, 1, 2, ..., seq_len-1]`，并复制batch_size份
4. 通过位置嵌入层得到 `position_embed`，形状与 `seqs_embed` 相同
5. 两者逐元素相加后通过Dropout层输出

### 2.3 Transformer编码器

本项目直接使用Paddle官方提供的 `nn.TransformerEncoderLayer` 和 `nn.TransformerEncoder` 实现Transformer编码器，无需手动实现多头注意力机制和前馈网络。

In [ ]:
# Transformer编码器定义 -  models/sasrec_model.py 第37-45行
self.encoder_layer = nn.TransformerEncoderLayer(
    d_model=hidden_units,       # 模型维度
    nhead=num_heads,            # 注意力头数
    dim_feedforward=hidden_units,  # 前馈网络维度
    dropout=dropout_rate,       # Dropout比例
)
self.encoder = nn.TransformerEncoder(
    encoder_layer=self.encoder_layer, 
    num_layers=num_blocks       # Transformer层数
)



### 2.4 因果掩码

为确保模型在预测位置i的下一个物品时，只能利用位置i及之前的信息，需要构造因果掩码（Causal Mask）。

In [ ]:
# 因果掩码生成 -  models/sasrec_model.py 第35行
self.subsequent_mask = paddle.triu(paddle.ones((max_len, max_len))) == 0

**掩码矩阵说明**：

使用 `paddle.triu` 生成上三角矩阵，然后取反得到下三角掩码。对于序列长度为4的情况，掩码矩阵如下：

```
[[True,  False, False, False],
 [True,  True,  False, False],
 [True,  True,  True,  False],
 [True,  True,  True,  True ]]
```

其中 `True` 表示对应位置可以参与注意力计算，`False` 表示被掩码屏蔽。这确保了位置i只能关注位置0到i的信息，实现了单向（因果）注意力。

---

## 3. 模型前向传播流程

### 3.1 训练阶段前向传播

训练时，模型接收历史序列、正样本序列和负样本序列作为输入，输出正负样本的预测分数。

In [ ]:
# 训练阶段前向传播 -  models/sasrec_model.py 第53-63行
def forward(self, log_seqs, pos_seqs, neg_seqs):
    # 1. 位置编码：将物品ID序列转换为嵌入表示
    seqs_embed = self.position_encoding(log_seqs)
    
    # 2. Transformer编码：通过自注意力机制提取序列特征
    log_feats = self.encoder(seqs_embed, self.subsequent_mask)

    # 3. 获取正负样本的物品嵌入
    pos_embed = self.item_emb(pos_seqs)
    neg_embed = self.item_emb(neg_seqs)

    # 4. 计算预测分数（内积）
    pos_logits = (log_feats * pos_embed).sum(axis=-1)
    neg_logits = (log_feats * neg_embed).sum(axis=-1)

    return pos_logits, neg_logits

**前向传播数据流**：

```
输入: log_seqs (batch_size, seq_len) - 用户历史物品ID序列
      pos_seqs (batch_size, seq_len) - 正样本物品ID序列（下一个物品）
      neg_seqs (batch_size, seq_len) - 负样本物品ID序列（随机采样）
                    ↓
        position_encoding: 物品嵌入 + 位置嵌入
                    ↓
        seqs_embed (batch_size, seq_len, hidden_units)
                    ↓
        Transformer Encoder (with causal mask)
                    ↓
        log_feats (batch_size, seq_len, hidden_units)
                    ↓
        与正负样本嵌入做逐元素乘法后求和（内积）
                    ↓
输出: pos_logits (batch_size, seq_len) - 正样本预测分数
      neg_logits (batch_size, seq_len) - 负样本预测分数
```

### 3.2 推理阶段前向传播

推理时，模型只使用序列最后一个位置的输出特征，与所有候选物品计算相似度分数。

In [ ]:
# 推理阶段前向传播 -  models/sasrec_model.py 第65-73行
def predict(self, log_seqs, item_indices):
    # 1. 位置编码和Transformer编码
    seqs = self.position_encoding(log_seqs)
    log_feats = self.encoder(seqs, self.subsequent_mask)

    # 2. 取序列最后一个位置的特征作为用户表示
    final_feat = log_feats[:, -1, :]
    
    # 3. 获取候选物品的嵌入
    item_embs = self.item_emb(paddle.to_tensor(item_indices, dtype="int64"))

    # 4. 计算用户表示与候选物品的相似度分数
    logits = item_embs.matmul(final_feat.unsqueeze(-1)).squeeze(-1)
    return logits

**推理流程说明**：

1. 与训练时相同，先进行位置编码和Transformer编码
2. 提取序列最后一个位置的特征向量 `final_feat`，形状为 `(batch_size, hidden_units)`，代表用户的即时兴趣
3. 获取所有候选物品的嵌入向量 `item_embs`，形状为 `(num_items, hidden_units)`
4. 通过矩阵乘法计算用户表示与每个候选物品的内积，得到预测分数

---

## 4. 损失函数设计

### 4.1 二元交叉熵损失

模型采用二元交叉熵损失函数（Binary Cross Entropy Loss），将推荐问题建模为二分类问题：正样本应该获得高分，负样本应该获得低分。

In [ ]:
# 自定义BCE损失函数 -  models/sasrec_model.py 第76-85行
class MyBCEWithLogitLoss(paddle.nn.Layer):
    def __init__(self):
        super(MyBCEWithLogitLoss, self).__init__()

    def forward(self, pos_logits, neg_logits, labels):
        return paddle.sum(
            -paddle.log(F.sigmoid(pos_logits) + 1e-24) * labels
            - paddle.log(1 - F.sigmoid(neg_logits) + 1e-24) * labels,
            axis=(0, 1),
        ) / paddle.sum(labels, axis=(0, 1))

**损失函数数学形式**：

$$\mathcal{L} = -\frac{1}{N}\sum_{i}\left[\log\sigma(s_{pos}^{(i)}) + \log(1-\sigma(s_{neg}^{(i)}))\right]$$

其中：
- $\sigma(\cdot)$ 为Sigmoid函数
- $s_{pos}$ 为正样本的预测分数
- $s_{neg}$ 为负样本的预测分数
- $N$ 为有效样本数量（排除padding位置）

**实现细节**：
- 使用 `labels` 参数作为mask，只计算有效位置的损失，padding位置（pos=0）不参与计算
- 添加 `1e-24` 作为数值稳定性的极小值，防止log(0)的情况
- 最终损失除以有效样本数进行归一化

---

## 5. 数据采样策略

### 5.1 序列采样逻辑

训练数据的构造采用滑动窗口方式，对每个用户的行为序列进行采样，生成历史序列、正样本和负样本三元组。

In [ ]:
# 序列采样函数 -  models/sasrec_ref/data.py 第16-41行
def sample_function(user_train, usernum, itemnum, batch_size, maxlen, result_queue=None, SEED=42):
    def sample():
        # 随机选择一个有足够交互记录的用户
        user = np.random.randint(1, usernum + 1)
        while len(user_train[user]) <= 1:
            user = np.random.randint(1, usernum + 1)

        # 初始化序列数组（padding为0）
        seq = np.zeros([maxlen], dtype=np.int32)  # 历史序列
        pos = np.zeros([maxlen], dtype=np.int32)  # 正样本序列
        neg = np.zeros([maxlen], dtype=np.int32)  # 负样本序列
        
        nxt = user_train[user][-1]  # 最后一个物品作为起始正样本
        idx = maxlen - 1

        ts = set(user_train[user])  # 用户已交互物品集合
        # 从后向前填充序列
        for i in reversed(user_train[user][:-1]):
            seq[idx] = i              # 当前物品作为历史
            pos[idx] = nxt            # 下一个物品作为正样本
            if nxt != 0:
                neg[idx] = random_neq(1, itemnum + 1, ts)  # 随机负采样
            nxt = i
            idx -= 1
            if idx == -1:
                break

        return (user, seq, pos, neg)

**采样流程说明**：

1. 随机选择一个用户，要求该用户至少有2条交互记录
2. 从用户行为序列的末尾开始，向前滑动窗口
3. 对于每个位置：
   - 当前物品作为历史序列的一部分
   - 下一个物品作为正样本（真实的下一个交互）
   - 从未交互物品中随机采样作为负样本
4. 序列不足 `maxlen` 长度时，前面用0进行padding

### 5.2 负采样策略

负采样时需要确保采样的物品不在用户的历史交互集合中。

In [ ]:
# 负采样函数 -  models/sasrec_ref/data.py 第9-13行
def random_neq(l, r, s):
    """在[l, r)范围内随机采样一个不在集合s中的整数"""
    t = np.random.randint(l, r)
    while t in s:
        t = np.random.randint(l, r)
    return t

### 5.3 多进程采样器

为提高数据加载效率，项目实现了基于多进程的异步采样器 `WarpSampler`，能够在训练的同时并行准备下一批数据。

In [ ]:
# 多进程采样器 -  models/sasrec_ref/data.py 第60-101行
class WarpSampler(object):
    def __init__(self, User, usernum, itemnum, batch_size=64, maxlen=10, n_workers=1):
        self.n_workers = n_workers
        if self.n_workers != 0:
            self.result_queue = Queue(maxsize=n_workers * 10)  # 结果队列
            self.processors = []
            for i in range(n_workers):
                # 启动多个采样进程
                self.processors.append(
                    Process(
                        target=sample_function,
                        args=(User, usernum, itemnum, batch_size, maxlen,
                              self.result_queue, np.random.randint(2e9)),
                    )
                )
                self.processors[-1].daemon = True
                self.processors[-1].start()

    def next_batch(self):
        """获取下一批训练数据"""
        if self.n_workers != 0:
            return self.result_queue.get()  # 从队列中获取预先采样好的数据
        return sample_function(...)  # 单进程模式下直接采样

**多进程采样优势**：

- 数据采样与模型训练并行执行，减少等待时间
- 通过队列实现生产者-消费者模式，保证数据供应的连续性
- 每个进程使用不同的随机种子，增加采样的多样性

---

## 6. 训练配置与优化

### 6.1 训练超参数

训练脚本 `train_sasrec.py` 提供了完整的超参数配置，可通过命令行参数进行调整。

In [ ]:
# 训练参数配置 -  train_sasrec.py 第25-61行
parser.add_argument("--epochs", type=int, default=200)         # 训练轮数
parser.add_argument("--batch_size", type=int, default=128)     # 批次大小
parser.add_argument("--max_len", type=int, default=200)        # 序列最大长度
parser.add_argument("--hidden_units", type=int, default=50)    # 嵌入维度
parser.add_argument("--num_heads", type=int, default=1)        # 注意力头数量
parser.add_argument("--num_blocks", type=int, default=2)       # Transformer块数量
parser.add_argument("--dropout", type=float, default=0.2)      # Dropout比例
parser.add_argument("--lr", type=float, default=0.001)         # 学习率
parser.add_argument("--l2_emb", type=float, default=0.0)       # L2正则化系数
parser.add_argument("--optimizer", type=str, default="AdamW")  # 优化器类型

**超参数配置总结**：

| 类别 | 参数 | 默认值 | 说明 |
|------|------|--------|------|
| 训练 | epochs | 200 | 训练轮数 |
| 训练 | batch_size | 128 | 批次大小 |
| 模型 | hidden_units | 50 | 嵌入和隐藏层维度 |
| 模型 | max_len | 200 | 序列最大长度 |
| 模型 | num_heads | 1 | 注意力头数 |
| 模型 | num_blocks | 2 | Transformer层数 |
| 正则化 | dropout | 0.2 | Dropout比例 |
| 正则化 | l2_emb | 0.0 | 嵌入层L2正则化系数 |
| 优化 | lr | 0.001 | 学习率 |
| 优化 | optimizer | AdamW | 优化器类型 |

### 6.2 优化器配置

项目支持多种优化器选择，使用PaddlePaddle官方提供的优化器实现。

In [ ]:
# 优化器配置 -  models/sasrec_ref/train.py 第26-37行
if args.optimizer == "Adam":
    optim = optimizer.Adam(
        parameters=model.parameters(), learning_rate=args.lr, grad_clip=clip
    )
elif args.optimizer == "Adagrad":
    optim = optimizer.Adagrad(
        parameters=model.parameters(), learning_rate=args.lr, grad_clip=clip
    )
elif args.optimizer == "AdamW":
    optim = optimizer.AdamW(
        parameters=model.parameters(), learning_rate=args.lr, grad_clip=clip
    )

**优化器说明**：

- **Adam**: 自适应学习率优化器，结合动量和RMSProp的优点
- **AdamW**: Adam的变体，使用解耦的权重衰减，更适合配合L2正则化使用
- **Adagrad**: 自适应梯度优化器，适合处理稀疏梯度的场景

默认使用AdamW优化器，学习率设为0.001。

### 6.3 L2正则化

为防止嵌入层过拟合，训练时对物品嵌入参数施加L2正则化。

In [ ]:
# L2正则化 -  models/sasrec_ref/train.py 第76-77行
for param in model.item_emb.parameters():
    loss += args.l2_emb * paddle.norm(param)

L2正则化项直接加到损失函数中，通过 `l2_emb` 系数控制正则化强度。默认值为0.0，即不使用L2正则化。

### 6.4 训练循环

完整的训练循环包含数据采样、前向传播、损失计算、反向传播和参数更新等步骤。

In [ ]:
# 训练循环核心代码 -  models/sasrec_ref/train.py 第60-81行
for epoch in range(start_epoch, args.epochs + 1):
    epoch_loss = 0
    for i_batch in range(num_batch):
        # 1. 获取一批训练数据
        u, seq, pos, neg = sampler.next_batch()
        u, seq, pos, neg = (
            paddle.to_tensor(u, dtype="int64"),
            paddle.to_tensor(seq, dtype="int64"),
            paddle.to_tensor(pos),
            paddle.to_tensor(neg),
        )
        
        # 2. 前向传播
        pos_logits, neg_logits = model(seq, pos, neg)

        # 3. 计算损失（排除padding位置）
        targets = (pos != 0).astype(dtype="float32")
        loss = criterion(pos_logits, neg_logits, targets)
        
        # 4. 添加L2正则化
        for param in model.item_emb.parameters():
            loss += args.l2_emb * paddle.norm(param)
        
        # 5. 反向传播和参数更新
        loss.backward()
        optim.step()
        optim.clear_grad()

### 6.5 模型保存

训练过程中会保存验证集上表现最好的模型，同时支持定期保存检查点。

In [ ]:
# 模型保存 -  models/sasrec_ref/train.py 第121-123行
def save_checkpoint(model, state, filename):
    state["state_dict"] = model.state_dict()
    paddle.save(state, filename)

保存的检查点包含：
- `state_dict`: 模型参数
- `epoch`: 当前训练轮次
- `optimizer`: 优化器状态
- `best_pair`: 最佳验证指标

最佳模型保存为 `SASRec_best.pth.tar`，定期检查点保存为 `SASRec_epoch_{n}.pth.tar`。

---

## 总结

---

**上一章**: [01_数据与实验设计报告.ipynb](./01_数据与实验设计报告.ipynb)  
**下一章**: [03_训练与评估报告.ipynb](./03_训练与评估报告.ipynb)